<a href="https://colab.research.google.com/github/Loopinlogix/Market_Analysis_Project-2/blob/main/Stock_Market_Project_2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Stock Market Analysis Project 2


## Intro to this Project

This notebook is all about digging into some historical stock market data. We're gonna do a bunch of things: grab the data, clean it up (get rid of weird errors and bad values), find any crazy outliers, check for duplicates, cook up some new features from the existing data, make sure everything's on the same scale, and then split it all up so we can eventually build some machine learning models.

Basically, we've got two main data files: one with general info about stocks (`historical_stocks.csv`) like where they're traded, their names, what industry they're in, etc., and another with the daily prices and trading volumes (`historical_stock_prices.csv`).

The whole point here is to take all that raw, messy stock info and turn it into something neat and organized, packed with useful features. This way, we'll have a solid dataset ready to go for training models to try and figure out what the stock market might do next.

In [ ]:

#Github

#Github
!apt-get install -y git
!git config --global user.email "crystal_macneil@hotmail.com"
!git config --global user.name "Crystal MacNeil"

!git clone https://github.com/Loopinlogix/Market_Analysis_Project-2.git
%cd Market_Analysis_Project-2
!ls


Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
git is already the newest version (1:2.34.1-1ubuntu1.17).
0 upgraded, 0 newly installed, 0 to remove and 3 not upgraded.
Cloning into 'Market_Analysis_Project-2'...
remote: Enumerating objects: 3, done.
remote: Counting objects: 100% (3/3), done.
remote: Total 3 (delta 0), reused 0 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (3/3), done.
/content/Market_Analysis_Project-2
README.md


In [3]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

print("=" * 60)
print("STEP 1: DATA COLLECTION")
print("=" * 60)

# Load datasets
# The CSV files are located directly in the /content/ directory.
stocks = pd.read_csv('historical_stocks.csv')
prices = pd.read_csv('historical_stock_prices.csv')

# Standardize column names
stocks.columns = stocks.columns.str.strip().str.lower()
prices.columns = prices.columns.str.strip().str.lower()

print("Prices head:")
print(prices.head())
print("\nStocks head:")
print(stocks.head())
print("\nPrices info:")
print(prices.info())

# Merge datasets on ticker
df = pd.merge(prices, stocks, on='ticker', how='left')

print("Merged DataFrame shape:", df.shape)

print("=" * 60)
print("STEP 2: ADVANCED CLEANING AND CLEANING ERRORS")
print("=" * 60)

#Fill missing categorical values with 'Unknown'
categorical_cols = ['exchange', 'name', 'sector', 'industry']
for col in categorical_cols:
    if col in df.columns:
        df[col] = df[col].fillna('Unknown')

print("\nMissing values after categorical fill:")
print(df.isnull().sum())

#Convert date column and handle errors
df['date'] = pd.to_datetime(df['date'], errors='coerce')

#Advanced imputation for numeric columns
numeric_cols = ['open', 'close', 'adj_close', 'low', 'high', 'volume']

#Sort by ticker and date to make time-based imputation meaningful
df.sort_values(by=['ticker', 'date'], inplace=True)

# Use forward fill and backward fill within each ticker group using transform
for col in numeric_cols:
    df[col] = df.groupby('ticker')[col].transform(lambda x: x.ffill().bfill())

# If any remaining NaNs (e.g., all values missing for a ticker), use interpolation using transform
for col in numeric_cols:
    df[col] = df.groupby('ticker')[col].transform(lambda x: x.interpolate(method='linear', limit_direction='both'))

#Finally, drop rows where date is still NaT (critical error)
df.dropna(subset=['date'], inplace=True)

print("\nMissing values after advanced numeric imputation and date cleaning:")
print(df.isnull().sum())

print(f"\nData type of 'date': {df['date'].dtype}")

print("\nSorted & cleaned DataFrame head:")
print(df.head())

print("=" * 60)
print("STEP 3: OUTLIER HANDLING")
print("=" * 60)

print("Descriptive statistics before outlier clipping:")
print(df[['close', 'volume']].describe())

for col in ['close', 'volume']:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR

    # Cap outliers instead of removing them
    df[col] = np.where(df[col] < lower, lower, df[col])
    df[col] = np.where(df[col] > upper, upper, df[col])

    print(f"Outliers capped for: {col}")

print("\nDescriptive statistics after outlier clipping:")
print(df[['close', 'volume']].describe())

print("=" * 60)
print("STEP 4: DUPLICATE CHECK")
print("=" * 60)

initial_rows = df.shape[0]
duplicate_count = df.duplicated(subset=['ticker', 'date']).sum()

print(f"Initial rows: {initial_rows}")
print(f"Duplicate rows found (ticker + date): {duplicate_count}")

df.drop_duplicates(subset=['ticker', 'date'], inplace=True)

removed = initial_rows - df.shape[0]
print(f"Duplicate rows removed: {removed}")
print(f"Final DataFrame shape: {df.shape}")


print("=" * 60)
print("STEP 5: FEATURE ENGINEERING")
print("=" * 60)

# Ensure sorted for rolling features
df.sort_values(by=['ticker', 'date'], inplace=True)

# Rolling window features (per ticker)
df['rolling_close_7'] = df.groupby('ticker')['close'].transform(lambda x: x.rolling(7).mean())
df['rolling_close_30'] = df.groupby('ticker')['close'].transform(lambda x: x.rolling(30).mean())

# Volatility (standard deviation)
df['volatility_7'] = df.groupby('ticker')['close'].transform(lambda x: x.rolling(7).std())
df['volatility_30'] = df.groupby('ticker')['close'].transform(lambda x: x.rolling(30).std())

# Daily returns
df['daily_return'] = df.groupby('ticker')['close'].pct_change()

# Price range (high - low)
df['price_range'] = df['high'] - df['low']

# Simple Moving Average (SMA)
df['sma_14'] = df.groupby('ticker')['close'].transform(lambda x: x.rolling(14).mean())

# Exponential Moving Average (EMA)
df['ema_14'] = df.groupby('ticker')['close'].transform(lambda x: x.ewm(span=14, adjust=False).mean())

# Relative Strength Index (RSI)
def compute_rsi(series, period=14):
    delta = series.diff()
    gain = delta.clip(lower=0)
    loss = -delta.clip(upper=0)
    avg_gain = gain.rolling(period).mean()
    avg_loss = loss.rolling(period).mean()
    rs = avg_gain / avg_loss
    return 100 - (100 / (1 + rs))

df['rsi_14'] = df.groupby('ticker')['close'].transform(compute_rsi)

print("Feature engineering complete. Displaying head:")
print(df.head())
print("=" * 60)
print("STEP 8: CONSOLIDATE FINAL MODELING DATASET")
print("=" * 60)

# Drop rows with remaining NaNs created by rolling windows
df_model = df.dropna().copy()

print(f"Final dataset shape after dropping remaining NaNs: {df_model.shape}")
display(df_model.head())

print("=" * 60)
print("STEP 6: CLEAN ROLLING FEATURE NA")
print("=" * 60)

# Drop rows with remaining NaNs created by rolling windows and RSI
df_model = df.dropna().copy()

print(f"Final dataset shape after dropping remaining NaNs: {df_model.shape}")
print(df_model.head())


print("=" * 60)
print("STEP 7: ENCODING CATEGORICAL VARIABLES")
print("=" * 60)

# We will one-hot encode exchange, sector, industry
categorical_to_encode = ['exchange', 'sector', 'industry']

df_model_encoded = pd.get_dummies(df_model, columns=categorical_to_encode, drop_first=True)

print("Shape before encoding:", df_model.shape)
print("Shape after encoding:", df_model_encoded.shape)
print(df_model_encoded.head())


# Drop rows with remaining NaNs created by rolling windows and RSI
df_model = df.dropna().copy()

print(f"Final dataset shape after dropping remaining NaNs: {df_model.shape}")
print(df_model.head())

print("=" * 60)
print("STEP 8: NORMALIZATION / STANDARDIZATION")
print("=" * 60)

# Identify numeric columns for scaling
numeric_cols_for_scaling = df_model_encoded.select_dtypes(include=[np.number]).columns.tolist()

# We will NOT scale the target 'close' yet, so remove it from the list
target = 'close'
numeric_cols_for_scaling = [col for col in numeric_cols_for_scaling if col != target]

scaler = StandardScaler()
df_model_encoded[numeric_cols_for_scaling] = scaler.fit_transform(df_model_encoded[numeric_cols_for_scaling])

print("Standardization complete.")
print(df_model_encoded[numeric_cols_for_scaling].describe().T.head())

print("=" * 60)
print("STEP 9: DATA SPLITTING")
print("=" * 60)

from sklearn.model_selection import train_test_split

# Example target variable (you can change this)
target = 'close'

# Features = everything except target
X = df_model_encoded.drop(columns=[target, 'ticker', 'date'])
y = df_model_encoded[target]

# First split: Train + Temp
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.30, random_state=42, shuffle=True
)

# Second split: Validation + Test
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, random_state=42, shuffle=True
)

print("Training set:", X_train.shape)
print("Validation set:", X_val.shape)
print("Test set:", X_test.shape)

print("=" * 60)
print("STEP 10: SAVE CLEAN DATA & SPLITS")
print("=" * 60)

# Save full cleaned dataset
df_model.to_csv('/content/clean_stock_data.csv', index=False)

# Save splits
X_train.to_csv('/content/X_train.csv', index=False)
X_val.to_csv('/content/X_val.csv', index=False)
X_test.to_csv('/content/X_test.csv', index=False)

y_train.to_csv('/content/y_train.csv', index=False)
y_val.to_csv('/content/y_val.csv', index=False)
y_test.to_csv('/content/y_test.csv', index=False)

print("All cleaned datasets and splits saved successfully.")

STEP 1: DATA COLLECTION
Prices head:
  ticker   open  close  adj_close    low   high     volume        date
0    AHH  11.50  11.58   8.493155  11.25  11.68  4633900.0  2013-05-08
1    AHH  11.66  11.55   8.471151  11.50  11.66   275800.0  2013-05-09
2    AHH  11.55  11.60   8.507822  11.50  11.60   277100.0  2013-05-10
3    AHH  11.63  11.65   8.544494  11.55  11.65   147400.0  2013-05-13
4    AHH  11.60  11.53   8.456484  11.50  11.60   184100.0  2013-05-14

Stocks head:
  ticker exchange                                    name             sector  \
0    PIH   NASDAQ  1347 PROPERTY INSURANCE HOLDINGS, INC.            FINANCE   
1  PIHPP   NASDAQ  1347 PROPERTY INSURANCE HOLDINGS, INC.            FINANCE   
2   TURN   NASDAQ                180 DEGREE CAPITAL CORP.            FINANCE   
3   FLWS   NASDAQ                 1-800 FLOWERS.COM, INC.  CONSUMER SERVICES   
4   FCCY   NASDAQ           1ST CONSTITUTION BANCORP (NJ)            FINANCE   

                     industry  
0  PROPERT

,ticker,open,close,adj_close,low,high,volume,date,exchange,name,...,industry,rolling_close_7,rolling_close_30,volatility_7,volatility_30,daily_return,price_range,sma_14,ema_14,rsi_14
1241,AAPL,0.578125,0.575893,0.026009,0.575893,0.578125,1294300.0,1981-01-26,NASDAQ,APPLE INC.,...,COMPUTER MANUFACTURING,0.576849,0.559152,0.012148,0.048808,-0.015267,0.002232,0.565210,0.572192,50.000000
1250,AAPL,0.575893,0.571429,0.025807,0.571429,0.575893,1294300.0,1981-01-27,NASDAQ,APPLE INC.,...,COMPUTER MANUFACTURING,0.579401,0.561086,0.007387,0.048076,-0.007752,0.004464,0.566645,0.572090,56.000010
1262,AAPL,0.555804,0.553571,0.025001,0.553571,0.555804,1294300.0,1981-01-28,NASDAQ,APPLE INC.,...,COMPUTER MANUFACTURING,0.574617,0.563318,0.011371,0.046009,-0.031250,0.002232,0.567602,0.569621,53.846135
1270,AAPL,0.535714,0.533482,0.024094,0.533482,0.535714,1294300.0,1981-01-29,NASDAQ,APPLE INC.,...,COMPUTER MANUFACTURING,0.569515,0.566071,0.019392,0.041278,-0.036290,0.002232,0.565051,0.564802,39.189208
1282,AAPL,0.508929,0.504464,0.022783,0.504464,0.508929,1294300.0,1981-01-30,NASDAQ,APPLE INC.,...,COMPUTER MANUFACTURING,0.558673,0.567485,0.030407,0.038204,-0.054393,0.004464,0.560746,0.556757,34.117664


STEP 6: CLEAN ROLLING FEATURE NA
Final dataset shape after dropping remaining NaNs: (3434954, 21)
     ticker      open     close  adj_close       low      high     volume  \
1241   AAPL  0.578125  0.575893   0.026009  0.575893  0.578125  1294300.0   
1250   AAPL  0.575893  0.571429   0.025807  0.571429  0.575893  1294300.0   
1262   AAPL  0.555804  0.553571   0.025001  0.553571  0.555804  1294300.0   
1270   AAPL  0.535714  0.533482   0.024094  0.533482  0.535714  1294300.0   
1282   AAPL  0.508929  0.504464   0.022783  0.504464  0.508929  1294300.0   

           date exchange        name  ...                industry  \
1241 1981-01-26   NASDAQ  APPLE INC.  ...  COMPUTER MANUFACTURING   
1250 1981-01-27   NASDAQ  APPLE INC.  ...  COMPUTER MANUFACTURING   
1262 1981-01-28   NASDAQ  APPLE INC.  ...  COMPUTER MANUFACTURING   
1270 1981-01-29   NASDAQ  APPLE INC.  ...  COMPUTER MANUFACTURING   
1282 1981-01-30   NASDAQ  APPLE INC.  ...  COMPUTER MANUFACTURING   

     rolling_close_7  ro